# 2.4 Sequence Unpacking and Extended Assignment

## Overview

Sequence unpacking is a deceptively compact component of Python's assignment semantics. In practice, unpacking works with **iterables**, not merely sequences, and can trigger iterator consumption, nested assignment, user-defined methods, and partial state changes. Extended unpacking adds starred targets that collect variable-length sections, while maintaining precise structural constraints.

For quantitative researchers and data scientists, these details matter when ingesting heterogeneous execution records, destructuring nested feature tuples, streaming market observations, and enforcing reliable financial data contracts. This notebook studies the grammar, execution model, and edge cases of unpacking, then builds an auditable execution-tape normalization and reconciliation pipeline.

### Learning Objectives

1. Explain the distinction between iterable unpacking, index access, and structural pattern matching.
2. Apply exact-length, nested, and starred assignment targets correctly.
3. Analyze `__iter__`, `__next__`, iterator exhaustion, and one-shot data sources.
4. Identify when target assignment can have side effects or partial failure.
5. Understand the difference between starred **targets** and starred **expressions** in calls or displays.
6. Inspect `UNPACK_SEQUENCE`, `UNPACK_EX`, and `ast.Starred` without relying on stable CPython bytecode.
7. Write explicit runtime contracts for heterogeneous trading records.
8. Validate an auditable, precision-preserving execution-tape pipeline.
9. Test financial reconciliation identities and mutation-preservation properties.
10. Evaluate the limitations of static typing, copying, and source-shape validation.

### Scope and Boundaries

This chapter addresses **unpacking as an assignment mechanism**. Ordinary multiple/chained assignment was developed in Section 2.3; full augmented assignment appears in Section 2.5. Pattern matching is discussed only to clarify its differences from unpacking. The financial case is trade-tape reconciliation, **not** a full portfolio NAV or realized-P&L engine.

### Notebook Conventions

Each conceptual topic follows **Theoretical Foundations → Advanced Code Example → Analytical Interpretation**. Code cells are designed to execute sequentially in a clean kernel. Intentional failures are caught and asserted rather than interrupting execution. Only trusted, deterministic synthetic financial records are used.

## 2.4.1 Formal Unpacking Semantics and Assignment Targets

### Theoretical Foundations

### Formal Model

An assignment such as `a, b = iterable` evaluates the right-hand side first, then obtains values from the object by iteration. It is **not** a promise to index the object at `[0]` and `[1]`. An iterator can satisfy an unpacking target even without supporting `len()` or `__getitem__()`.

| Construct | Meaning | Important Constraint |
|---|---|---|
| `a, b = source` | Exact unpacking | Exactly two yielded items |
| `(a, b) = source` | Same assignment target grouping | Parentheses do not make a tuple result mandatory |
| `[a, b] = source` | List-shaped target pattern | Also works with an iterable |
| `a, (b, c) = source` | Recursive targets | Each nested iterable has its own arity requirement |
| `a, *middle, z = source` | Extended unpacking | At least two items; `middle` is a **list** |
| `a, *rest = source` | Prefix followed by remainder | At least one item |
| `*rest, = source` | Entire source into starred target | `rest` is a list |

### Assignment Semantics

- The complete right-hand-side expression is evaluated before assignments begin.
- Unpacking relies on the **iteration protocol**, including user-defined iterables and generators.
- Target assignments proceed **left to right**, including inside recursive targets.
- Unpacking itself can raise `TypeError` or `ValueError` before or between target writes.
- A right-hand side may be exhausted, may raise an exception, or may produce side effects while being consumed.

### Do Not Confuse

**Iterable unpacking** assigns values; **subscription** reads by an index/key; **pattern matching** tests a subject against a pattern, can fail without raising an arity error, and obeys different semantic rules.

In [1]:
from collections.abc import Iterable

# Unpacking works with tuples, lists, ranges, and generators.
trade_id, ticker = ("F001", "AAPL")
quantity, price = [100, 185.25]
low, high = range(10, 12)
first, second = (number * 2 for number in (3, 5))

assert (trade_id, ticker) == ("F001", "AAPL")
assert (quantity, price) == (100, 185.25)
assert (low, high) == (10, 11)
assert (first, second) == (6, 10)

# The *target* receives a new list, even if the source is a tuple.
head, *middle, tail = ("OPEN", "AAPL", "MSFT", "CLOSE")
assert head == "OPEN" and tail == "CLOSE"
assert middle == ["AAPL", "MSFT"]
assert type(middle) is list

print("Iterable-based unpacking verified.")
print("Intermediate assets:", middle)
print("Iterable?", isinstance((x for x in range(2)), Iterable))

Iterable-based unpacking verified.
Intermediate assets: ['AAPL', 'MSFT']
Iterable? True


### **Analytical Interpretation**

**1. Iterable Semantics.** A generator can populate target names without supporting integer indexing. Unpacking does not require a sequence object.

**2. RHS-to-Targets Separation.** The evaluated iterable provides values; the target shape determines how values are bound.

**3. Starred Result Type.** `middle` is a list, even when the source is an immutable tuple; this is important for aliasing and defensive copying.

**4. Exact Arity.** Ordinary unpacking rejects surplus or missing yielded items rather than silently discarding them.

**5. Quant Implication.** When consuming broker records, input shape is an executable contract. Extra or missing fields must be handled deliberately.

## 2.4.2 Iterator Protocol, Arity Validation, and Consumption

### Theoretical Foundations

### Iteration, Not Positional Indexing

Python ordinarily obtains an iterator with `iter(source)` and repeatedly requests values via `next(iterator)` during unpacking. User-defined objects can make these operations observable. Some built-in iterable types have optimized implementations, but the *language-level result* must follow unpacking semantics.

### Arity Contracts

- `a, b = it` requires **exactly two** values; fewer or more raise `ValueError`.
- Determining that an iterable is too long may require requesting an **additional** item.
- If `iter(source)` is not supported, unpacking raises `TypeError`.
- Iterators may be stateful. A failed unpack can still consume values and perform side effects.
- `__length_hint__` is an optimization hint, **not** an exact-length or safety guarantee.

### Production Consideration

Repeated or unbounded inputs can impose computational, memory, and operational costs. Validating shape on one-shot iterables should be done with deliberate resource bounds, not by assuming they are finite.

In [2]:
class TracedIterator:
    """An iterator exposing every request for another item."""

    def __init__(self, values):
        self._values = iter(values)
        self.events = []

    def __iter__(self):
        self.events.append("iter")
        return self

    def __next__(self):
        self.events.append("next")
        return next(self._values)


exact = TracedIterator(["AAPL", 100])
ticker, shares = exact
assert (ticker, shares) == ("AAPL", 100)
assert exact.events.count("next") >= 2

# A surplus item causes a length mismatch; the iterator is consumed.
excess = TracedIterator(["AAPL", 100, "EXTRA", "LATER"])
try:
    ticker, shares = excess
except ValueError as exc:
    assert "too many" in str(exc).lower()
else:
    raise AssertionError("Expected surplus-item failure")

# The first three items have been requested; "LATER" remains available.
assert next(excess) == "LATER"

short = iter(["ONLY_ONE"])
try:
    a, b = short
except ValueError:
    pass
else:
    raise AssertionError("Expected insufficient-item failure")

print("Exact-unpack trace:", exact.events)
print("Excess-unpack trace:", excess.events)

Exact-unpack trace: ['iter', 'next', 'next', 'next']
Excess-unpack trace: ['iter', 'next', 'next', 'next', 'next']


### **Analytical Interpretation**

**1. Protocol Execution.** The iterator's methods are invoked as values are requested. The exact number of internal implementation calls should not be treated as a cross-implementation bytecode contract.

**2. Surplus Detection.** Detecting excess values requires observing that more values exist than available targets; a failed unpack can consume data.

**3. One-Shot Semantics.** The iterator's position changes even when assignment fails. Retrying the same unpack against the same iterator does not restore its earlier state.

**4. Input Reliability.** Streaming market-data ingestion must distinguish validation of an incoming record from the irreversibility of consuming a live feed.

**5. Engineering Rule.** Normalize trusted bounded records into stable objects before downstream state changes when replayability is required.

## 2.4.3 Nested Destructuring, Mixed Targets, and Structural Contracts

### Theoretical Foundations

### Recursive Target Lists

Nested targets recursively unpack the corresponding value. For example, `(symbol, (quantity, price)) = record` requires two outer items and exactly two inner numeric items. The iterable's concrete type at each level is not prescribed.

### Mixed Target Types

A target list can include ordinary names, object attributes, and subscription targets. Every target may have its own assignment behavior and may call user-defined code. The order of those target assignments is observable.

### Shape Is Not Type Safety

Successful unpacking does not establish that `quantity` is an integer, `price` is finite, or `symbol` is a meaningful security identifier. Domain validation must follow unpacking (or occur within a dedicated normalized data model).

### Future Coverage

Deep iterable-unpacking patterns and assignments are distinct from `match`/`case` pattern matching, even where their visual syntax resembles one another.

In [3]:
from dataclasses import dataclass

@dataclass
class MutableFill:
    """Simple mutable destination for a nested assignment."""
    ticker: str = ""
    quantity: int = 0
    price: float = 0.0


raw_record = ("F003", ["JPM", (75, 150.25)])
fill_id, (ticker, (quantity, price)) = raw_record
assert (fill_id, ticker, quantity, price) == (
    "F003", "JPM", 75, 150.25
)

mut = MutableFill()
values = ("AAPL", (100, 191.50))
mut.ticker, (mut.quantity, mut.price) = values
assert (mut.ticker, mut.quantity, mut.price) == ("AAPL", 100, 191.5)

# Recursive destructuring can use heterogeneous nested iterable types.
record = ["MSFT", iter([50, 405.75])]
name, (qty, execution_price) = record
assert (name, qty, execution_price) == ("MSFT", 50, 405.75)

print("Nested fill:", fill_id, ticker, quantity, price)
print("Assigned mutable destination:", mut)

Nested fill: F003 JPM 75 150.25
Assigned mutable destination: MutableFill(ticker='AAPL', quantity=100, price=191.5)


### **Analytical Interpretation**

**1. Recursive Shape.** Nested unpacking enforces the structure separately at each nesting depth.

**2. Mixed Sources.** Lists, tuples, and iterators may appear at different depths; structural matching is based on iteration behavior.

**3. Attribute Targets.** Assignments to `mut` change the existing object; they are not merely new name bindings.

**4. Domain Constraints.** Correct unpacking only establishes a shape. It does not prove that financial quantities, prices, venues, or IDs are valid.

**5. State Consistency.** When target assignments mutate an existing object, prefer validation before mutation or immutable object construction.

## 2.4.4 Starred Targets and Extended Iterable Unpacking (PEP 3132)

### Theoretical Foundations

### Extended Assignment

PEP 3132 introduced starred targets in assignment target lists. A starred target gathers the items not assigned to explicitly positioned targets.

```python
first, *middle, last = source
*prefix, last = source
first, *suffix = source
*all_items, = source
```

### Guarantees and Restrictions

1. At most **one starred target per single unpacking target list**; distinct nested target lists can each contain their own starred targets.
2. Required targets before and after the star must be satisfied.
3. The starred target receives a **list**, possibly empty.
4. The starred target may itself be an attribute or subscription target if allowed by assignment grammar; its assignment can trigger side effects.
5. A standalone starred target is invalid: use `*items, = source` rather than `*items = source`.
6. A starred assignment target and an unpacking `*expression` in a list display or function call are **different syntactic roles**.

### Computational Consideration

Gathering the starred remainder into a list requires memory proportional to the number of collected elements. Avoid unbounded generators in starred assignments.

In [4]:
observations = [100.0, 102.0, 104.0, 103.0, 108.0]
first, *middle, last = observations
assert first == 100.0 and last == 108.0
assert middle == [102.0, 104.0, 103.0]

first, *empty, last = [100.0, 105.0]
assert empty == []

*everything, = ("AAPL", "MSFT")
assert everything == ["AAPL", "MSFT"]

# The nested target lists independently support one star each.
(prefix, *rest), (head, *tail) = (
    ("AAPL", "MSFT", "JPM"),
    (100, 200, 300),
)
assert rest == ["MSFT", "JPM"]
assert tail == [200, 300]

# A starred target can populate a mutable subscription target.
result = {}
start, *result["core"], end = (1, 2, 3, 4)
assert (start, end) == (1, 4)
assert result["core"] == [2, 3]

# Invalid grammar is checked without executing the source string.
invalid_sources = [
    "a, *b, *c = [1, 2, 3]",
    "*items = range(3)",
]
for source in invalid_sources:
    try:
        compile(source, "<invalid-starred-assignment>", "exec")
    except SyntaxError:
        pass
    else:
        raise AssertionError(f"Unexpected valid syntax: {source}")

print("Starred target results:", middle, empty, everything, result)

Starred target results: [102.0, 104.0, 103.0] [] ['AAPL', 'MSFT'] {'core': [2, 3]}


### **Analytical Interpretation**

**1. Variable-Length Capture.** A starred target provides a flexible remainder while preserving required prefix/suffix fields.

**2. Empty Remainders.** The starred target is always a list, including when no intermediate elements remain.

**3. Nested Stars.** Multiple starred targets are possible only when they belong to different nested target lists.

**4. Side Effects.** The starred destination may be a subscript or attribute; the collected list may be stored through a custom assignment operation.

**5. Financial Record Design.** Prefix/suffix requirements and optional trailing tags are appropriate use cases; a star should not disguise an ambiguous or undocumented schema.

## 2.4.5 Lazy Iterables, Generators, and One-Pass Data Pipelines

### Theoretical Foundations

### Generators as Unpacking Sources

Generators produce values on demand. Unpacking forces the necessary generator evaluations, often including additional work to confirm exact length. After consumption, the generator does not automatically restart.

### Resource Implications

- Exact unpacking is appropriate for fixed-size records.
- A starred target consumes the **remainder** into a list; using it on a very large or infinite stream is not a safe bounded operation.
- `itertools.islice()` permits bounded extraction from a stream but also consumes its input.
- `itertools.tee()` can duplicate iterator views, but its buffering requirements may grow when views advance at different rates.
- Do not infer iterable replayability from the annotation `Iterable[T]`.

### Data Engineering Implications

Separate finite record parsing from continuous market feeds. A stream adapter should turn each event into one bounded record before applying arity checks.

In [5]:
from itertools import islice


def observed_generator(values, trace):
    for value in values:
        trace.append(f"yield:{value}")
        yield value


trace = []
source = observed_generator(["AAPL", 100, 185.50], trace)
ticker, shares, price = source
assert (ticker, shares, price) == ("AAPL", 100, 185.5)
assert trace == ["yield:AAPL", "yield:100", "yield:185.5"]
assert list(source) == []  # One-shot iterator exhausted.

# Consume a bounded prefix without materializing an entire feed.
feed = (f"F{n:03d}" for n in range(1, 100_000))
first_three = tuple(islice(feed, 3))
next_event = next(feed)
assert first_three == ("F001", "F002", "F003")
assert next_event == "F004"

# Starred capture is safe only when the iterable is known to be finite.
head, *tail = iter(range(5))
assert head == 0 and tail == [1, 2, 3, 4]

print("Generator trace:", trace)
print("Bounded prefix:", first_three)

Generator trace: ['yield:AAPL', 'yield:100', 'yield:185.5']
Bounded prefix: ('F001', 'F002', 'F003')


### **Analytical Interpretation**

**1. Lazy Evaluation.** The generator executes only when unpacking requests its elements.

**2. Exhaustion.** Once a generator has been consumed, subsequent iteration resumes from its current position, not its start.

**3. Bounded Parsing.** `islice()` controls how many items are requested; it does not reset the source or supply transactional guarantees.

**4. Materialization Cost.** A starred target builds a list of remaining values; this may be a serious memory issue for unbounded feeds.

**5. Quant Pipeline.** Real-time event ingestion should treat one-shot input consumption as a stateful operation and use bounded parsing and replay infrastructure where needed.

## 2.4.6 Unpacking Mappings, Strings, Enumeration, and Zipped Records

### Theoretical Foundations

### What Does an Iterable Yield?

Unpacking uses the iterable's **iteration values**, not some universal interpretation of its contents.

- Iterating a dictionary yields its **keys**, not its values.
- Iterating a string yields Unicode code points as one-character strings; this is not equivalent to splitting user-perceived grapheme clusters.
- `enumerate()` yields `(index, value)` pairs suitable for nested unpacking.
- `zip()` yields tuples of corresponding elements and normally stops at the shortest input; `strict=True` raises on length mismatch (Python 3.10+).
- `dict.items()` yields `(key, value)` pairs, supporting direct unpacking in loops.

### Important Distinction

`a, b = mapping` assigns the first two keys, whereas `a, b = mapping.values()` assigns values. `for key, value in mapping.items()` unpacks each yielded pair separately.

### Finite Schema Guarantees

Use `zip(..., strict=True)` when sources must be aligned. Without `strict=True`, truncation can silently discard data from an input series.

In [6]:
from math import fsum

portfolio_weights = {"AAPL": 0.45, "MSFT": 0.35, "JPM": 0.20}
first_key, second_key, third_key = portfolio_weights
assert (first_key, second_key, third_key) == ("AAPL", "MSFT", "JPM")

# Dictionary .items() yields tuples to unpack in the loop target.
for ticker, weight in portfolio_weights.items():
    assert isinstance(ticker, str)
    assert isinstance(weight, float)

# enumerate yields (index, item) pairs.
rows = list(enumerate(("BUY", "SELL"), start=1))
for order_number, side in rows:
    assert order_number in (1, 2)
    assert side in ("BUY", "SELL")

prices = [190.0, 410.0, 205.0]
tickers = list(portfolio_weights)
weighted_prices = {
    ticker: weight * price
    for (ticker, weight), price in zip(
        portfolio_weights.items(), prices, strict=True
    )
}
assert abs(fsum(weighted_prices.values()) - 270.0) < 1e-10

try:
    list(zip(["AAPL", "MSFT"], [100.0], strict=True))
except ValueError:
    pass
else:
    raise AssertionError("Strict zip failed to detect misalignment")

# Strings yield Unicode code points, not words or full grapheme clusters.
first_char, second_char = "AB"
assert (first_char, second_char) == ("A", "B")

print("Mapped weighted values:", weighted_prices)
print("Enumerated sides:", rows)

Mapped weighted values: {'AAPL': 85.5, 'MSFT': 143.5, 'JPM': 41.0}
Enumerated sides: [(1, 'BUY'), (2, 'SELL')]


### **Analytical Interpretation**

**1. Mapping Semantics.** Direct dictionary unpacking yields keys. Use `.values()` or `.items()` when the schema requires values or key–value pairs.

**2. Record Alignment.** `zip(..., strict=True)` checks that paired inputs are equally long, preventing selected silent truncation bugs.

**3. Nested Loop Targets.** A loop can destructure each `(key, value)` pair and a separately supplied price in one coherent iteration.

**4. Unicode Semantics.** Unpacking a string does not necessarily produce visually distinct characters as users perceive them.

**5. Financial Caution.** The weighted-price calculation is a **weighted price sum**, not a meaningful portfolio return; align variables and their units before interpreting any aggregate.

## 2.4.7 Starred Assignment vs. Iterable Expansion in Expressions and Calls

### Theoretical Foundations

### Three Different Contexts

**Starred assignment target**: `head, *rest = iterable` captures a list.

**Starred expression in a display**: `[*left, *right]` expands iterables into a new list (or other supported display context).

**Starred call arguments**: `fn(*iterable)` supplies positional arguments. A double-starred mapping expansion, `fn(**mapping)`, supplies keyword arguments.

These constructs use related surface syntax but obey different grammar and type rules.

### Domain Engineering Considerations

- `*iterable` in a call can duplicate or reorder meaning if a positional API is unstable; keyword arguments are often clearer for financial parameters.
- `**mapping` is subject to keyword-name validity and duplicate keyword rules.
- `[ *values ]` is a newly constructed **outer** list but does not recursively clone nested mutable elements.
- List/tuple concatenation via star expansion does not imply numerical addition or a shared data model.

In [7]:
from math import fsum

sectors = ("Technology", "Financials")
additional = ["Energy", "Bonds"]
combined = [*sectors, *additional]
assert combined == ["Technology", "Financials", "Energy", "Bonds"]

# A starred assignment target CAPTURES the remainder.
first_sector, *remaining_sectors = combined
assert first_sector == "Technology"
assert remaining_sectors == ["Financials", "Energy", "Bonds"]


def weighted_sum(prices, weights):
    """Return a price-weight sum, not a return statistic."""
    return fsum(p * w for p, w in zip(prices, weights, strict=True))


arguments = ([100.0, 200.0], [0.25, 0.75])
from_positional_expansion = weighted_sum(*arguments)
from_keyword_expansion = weighted_sum(
    **{"prices": arguments[0], "weights": arguments[1]}
)
assert from_positional_expansion == 175.0
assert from_keyword_expansion == 175.0

# Outer expansion does not deep-copy nested mutable objects.
nested = [["AAPL"], ["MSFT"]]
expanded = [*nested]
assert expanded is not nested
assert expanded[0] is nested[0]

print("Starred capture:", remaining_sectors)
print("Expanded call result:", from_keyword_expansion)

Starred capture: ['Financials', 'Energy', 'Bonds']
Expanded call result: 175.0


### **Analytical Interpretation**

**1. Context Distinction.** The same `*` character can capture in assignment or expand in a call or display.

**2. Return Types.** An assignment-star produces a list for the captured values; an expression-star provides expanded values to the surrounding construct.

**3. API Stability.** Explicit keyword arguments can prevent positional misinterpretation when passing financial assumptions.

**4. Shallow Structure.** Expansion into a new list does not isolate nested mutable inputs.

**5. Quant Interpretation.** Never infer financial meaning from a syntactically convenient expansion; prices, weights, and returns are different concepts requiring explicit contracts.

## 2.4.8 Iterable Unpacking vs. Structural Pattern Matching

### Theoretical Foundations

### Similar Syntax, Different Operations

Unpacking assignment binds values and raises exceptions when the source has an invalid shape. Structural pattern matching (`match`/`case`, Python 3.10+) **tests** candidate patterns and can select a fallback case.

### Key Differences

| Feature | Assignment unpacking | Sequence pattern in `match` |
|---|---|---|
| Purpose | Assign targets | Test and deconstruct a subject |
| Wrong length | Usually `ValueError` | Pattern does not match |
| Allowed input | General iterable | Supported sequence-pattern subjects (not arbitrary iterators; strings/bytes excluded) |
| Starred capture | List | List for starred sequence capture |
| Failure behavior | May leave side effects/partial assignments | No match branch selected; binding details in failed compound matches require care |
| Type validation | Separate checks needed | Pattern may express some structural/type constraints |

A generator can be unpacked by assignment but is not automatically a sequence-pattern subject. Never use `match` merely as a substitute for robust input validation.

In [8]:
# Pattern matching is supported on Python 3.10+.
record = ("AAPL", 100, 185.50)

match record:
    case (str() as ticker, int() as shares, float() as price) if shares > 0:
        parsed = (ticker, shares, price)
    case _:
        raise ValueError("Unsupported trade record")

assert parsed == ("AAPL", 100, 185.50)

# Unpacking has different shape-failure semantics.
try:
    a, b = record
except ValueError:
    pass
else:
    raise AssertionError("Expected exact-unpacking failure")

match ("AAPL", 100, 185.50):
    case (symbol, quantity):
        classification = "TWO_FIELD"
    case (symbol, quantity, execution_price):
        classification = "THREE_FIELD"
    case _:
        classification = "UNKNOWN"

assert classification == "THREE_FIELD"

# An iterator can be unpacked, but it does not satisfy sequence patterns.
source = iter(["MSFT", 50])
match source:
    case [symbol, quantity]:
        generator_pattern = "MATCH"
    case _:
        generator_pattern = "NO_MATCH"
assert generator_pattern == "NO_MATCH"

print("Matched trade:", parsed)
print("Generator sequence pattern:", generator_pattern)

Matched trade: ('AAPL', 100, 185.5)
Generator sequence pattern: NO_MATCH


### **Analytical Interpretation**

**1. Assignment Failure.** Exact unpacking throws on a shape mismatch; pattern matching falls through to another `case`.

**2. Subject Protocol.** Structural sequence patterns do not apply to every object that can be iterated. Treat this as a substantive semantic distinction.

**3. Guard Logic.** A pattern can enforce some type and shape conditions, but domain-specific constraints still need validation.

**4. Binding Semantics.** Pattern capture names bind in the surrounding scope on success. Avoid relying on unspecified bindings after a failed partial match.

**5. Financial Data Ingestion.** Use a single documented parser that defines accepted record shapes and explicit failure behavior; choose syntax based on those requirements, not appearance.

## 2.4.9 Failure Boundaries, Nested Partial Assignment, and Side Effects

### Theoretical Foundations

### Failure Modes

1. **Top-level mismatch:** `a, b = iterable` can fail during initial collection before any top-level target is assigned.
2. **Nested mismatch:** `a, (b, c) = source` may assign `a` successfully before the nested unpack fails.
3. **Target-side failure:** even after successful unpacking, a later attribute or subscription target can reject its value.
4. **Iterator-side effects:** iteration may consume values or modify external state before failing.

### Atomicity and Rollback

Python unpacking is **not a general transaction**. A caught exception does not automatically undo earlier target writes or iterator consumption. For financial state, parse and validate a bounded temporary record before publishing any authoritative updates.

### Diagnostic Discipline

Record *before* and *after* state, distinguish source-consumption failures from target-assignment failures, and test both cases explicitly.

In [9]:
# Top-level shape mismatch does not rebind these simple name targets.
a, b = "old-a", "old-b"
try:
    a, b = [1]
except ValueError:
    pass
else:
    raise AssertionError("Expected exact-unpacking failure")
assert (a, b) == ("old-a", "old-b")

# A nested unpacking failure can leave an EARLIER target rebound.
first, second, third = "old-first", "old-second", "old-third"
try:
    first, (second, third) = ("new-first", [99])
except ValueError:
    pass
else:
    raise AssertionError("Expected nested-unpacking failure")
assert first == "new-first"
assert (second, third) == ("old-second", "old-third")

# Target methods can reject later assignments after earlier side effects.
class RejectSecond:
    def __init__(self):
        self.data = {}

    def __setitem__(self, key, value):
        if key == "reject":
            raise ValueError("Rejected target")
        self.data[key] = value


state = RejectSecond()
try:
    state["accepted"], state["reject"] = (10, 20)
except ValueError:
    pass
else:
    raise AssertionError("Expected target-assignment failure")
assert state.data == {"accepted": 10}

print("Nested partial write:", first, second, third)
print("Target-side partial state:", state.data)

Nested partial write: new-first old-second old-third
Target-side partial state: {'accepted': 10}


### **Analytical Interpretation**

**1. Mismatch Timing.** A top-level unpack can fail before its simple names are assigned; a nested mismatch can occur after earlier outer targets are updated.

**2. Partial State.** The first nested target is rebound even though the full assignment raises an exception.

**3. Target Protocols.** Custom item assignment can reject a later value after an earlier item write has succeeded.

**4. No Automatic Rollback.** Exception handling catches the failure but does not restore modified objects or rewind iterators.

**5. Quant Design.** Parse an entire event into a validated immutable record before updating cash, holdings, or risk metadata.

## 2.4.10 AST, Starred Targets, and CPython Bytecode

### Theoretical Foundations

### Structural Analysis

In Python's AST, destructuring is represented through `ast.Tuple` or `ast.List` nodes in a `Store` context, containing nested targets. Starred targets appear as `ast.Starred` with an assignment context; they are distinguishable from starred expansion in function calls.

### Bytecode View

In CPython, disassembly often uses `UNPACK_SEQUENCE` for fixed-size destructuring and `UNPACK_EX` for extended destructuring. The latter encodes counts of mandatory targets before and after the star. **Bytecode formats and exact instruction sequences are implementation/version dependent.**

### Static Analysis Limitations

AST inspection identifies structural targets but cannot prove whether a runtime iterable is long enough, whether iterator methods raise, or whether assignment into an attribute has side effects.

In [10]:
import ast
import dis

source = """
a, b = iterable_source
first, *middle, last = iterable_source
symbol, (shares, price) = nested_record
"""
tree = ast.parse(source)

for statement in tree.body:
    assert isinstance(statement, ast.Assign)
    target = statement.targets[0]
    print(f"line={statement.lineno}, target={type(target).__name__}")
    print(ast.dump(target, include_attributes=False))

starred_nodes = [
    node for node in ast.walk(tree) if isinstance(node, ast.Starred)
]
assert len(starred_nodes) == 1
assert isinstance(starred_nodes[0].ctx, ast.Store)


def fixed_unpack(source):
    a, b = source
    return a, b


def extended_unpack(source):
    a, *middle, b = source
    return a, middle, b


fixed_ops = [ins.opname for ins in dis.get_instructions(fixed_unpack)]
extended_ops = [ins.opname for ins in dis.get_instructions(extended_unpack)]
assert "UNPACK_SEQUENCE" in fixed_ops
assert "UNPACK_EX" in extended_ops

print("Fixed unpack instruction:", [op for op in fixed_ops if "UNPACK" in op])
print("Extended unpack instruction:", [op for op in extended_ops if "UNPACK" in op])

line=2, target=Tuple
Tuple(elts=[Name(id='a', ctx=Store()), Name(id='b', ctx=Store())], ctx=Store())
line=3, target=Tuple
Tuple(elts=[Name(id='first', ctx=Store()), Starred(value=Name(id='middle', ctx=Store()), ctx=Store()), Name(id='last', ctx=Store())], ctx=Store())
line=4, target=Tuple
Tuple(elts=[Name(id='symbol', ctx=Store()), Tuple(elts=[Name(id='shares', ctx=Store()), Name(id='price', ctx=Store())], ctx=Store())], ctx=Store())
Fixed unpack instruction: ['UNPACK_SEQUENCE']
Extended unpack instruction: ['UNPACK_EX']


### **Analytical Interpretation**

**1. Target Representation.** `ast.Tuple`/`ast.List` organize nested assignment targets; `ast.Starred` records extended capture.

**2. Compile-Time Shape.** The compiler knows the number of required targets but cannot generally know the length of arbitrary runtime input.

**3. Bytecode Evidence.** CPython distinguishes fixed-size and extended-unpacking operations in its disassembly.

**4. Portability.** Do not build application behavior around a fixed bytecode instruction sequence; rely on Python's defined language semantics.

**5. Research Tooling.** AST-based linting can find dubious target structures, but actual financial record validation still needs runtime tests.

## 2.4.11 Type Contracts, Performance, and Unpacking Pitfalls

### Theoretical Foundations

### Static Typing and Shape

Annotations such as `tuple[str, int, float]` describe fixed-length heterogeneous records. `Sequence[object]` allows more variation but loses type information and fixed-length guarantees; runtime validation is still necessary for external data.

`NamedTuple` and frozen dataclasses can provide named fields and predictable interfaces after data normalization. Destructuring typed records can be convenient, but positional fields are fragile when a provider changes the schema.

### Common Failure Modes

1. Implicitly assuming a generator is replayable.
2. Assuming a starred target produces a tuple instead of a list.
3. Treating unpacked mapping keys as values.
4. Silently truncating aligned inputs with ordinary `zip`.
5. Unpacking raw strings character-by-character by mistake.
6. Applying a star to an infinite feed, causing unbounded consumption.
7. Assuming nested unpacking is rollback-safe.
8. Confusing `match` patterns with assignment targets.
9. Hiding financial units through overly compact positional records.
10. Assuming syntax-level success implies valid prices, fees, dates, or symbols.

### Performance

Direct fixed-size unpacking is generally simple and efficient, but benchmarks should consider object creation, generator behavior, conversion cost, and the memory cost of starred capture. The clarity of a documented schema is more important than speculative micro-optimizations.

In [11]:
from typing import NamedTuple


class ExecutionTuple(NamedTuple):
    ticker: str
    quantity: int
    price_usd: float


record = ExecutionTuple("AAPL", 100, 185.25)
symbol, shares, execution_price = record
assert (symbol, shares, execution_price) == ("AAPL", 100, 185.25)
assert record.ticker == "AAPL"

# Unpacking a dictionary selects keys, a common silent schema bug.
record_map = {"ticker": "AAPL", "quantity": 100}
a, b = record_map
assert (a, b) == ("ticker", "quantity")
assert (a, b) != ("AAPL", 100)

# A generator is not reusable unless the producer recreates it.
source = (value for value in [100, 200])
p, q = source
assert (p, q) == (100, 200)
try:
    p, q = source
except ValueError:
    pass
else:
    raise AssertionError("Expected exhausted-generator failure")

print("Typed record:", record)
print("Mapping keys, not values:", a, b)

Typed record: ExecutionTuple(ticker='AAPL', quantity=100, price_usd=185.25)
Mapping keys, not values: ticker quantity


### **Analytical Interpretation**

**1. Typed Records.** `NamedTuple` gives positional unpacking and named access; it improves clarity but does not enforce the types of every field at runtime.

**2. Key–Value Confusion.** Mapping unpacking can silently return key names that have the wrong business meaning.

**3. Iterator Lifetime.** Reusing an exhausted generator violates the assumed input contract even though the generator object still exists.

**4. Performance Perspective.** Avoid materializing more data than needed, but do not substitute micro-optimizations for clear schemas and reproducible processing.

**5. Financial Safety.** Positional record structure must be documented and validated before its values enter accounting or risk calculations.

## 2.4.12 Practical Application — Schema-Aware Execution Tape Normalization and Reconciliation

### Project Overview

**Project:** *Extended Unpacking for Heterogeneous Trade Events*

### Research Objective

Build a deterministic, precision-preserving trade-tape processor that turns variable-length nested records into validated immutable execution events. Reconcile per-asset signed quantities, venue-level notionals, and buy/sell cash-flow identities. The objective is to demonstrate the reliability implications of **nested unpacking, starred capture, schema validation, and one-pass ingestion**, without implementing a portfolio accounting ledger.

### Raw Record Schema

```text
(
    event_id,
    (ticker, side),
    (quantity, execution_price_usd),
    (fee_usd, venue),
    *optional_tags
)
```

An event has four mandatory outer fields, plus zero to three trailing string tags. Each mandated pair must contain exactly two entries. Price and fee data are canonical USD decimal strings with exactly two digits after the decimal point.

### Workflow

```text
Bounded, finite event iterable
           |
           v
 Validate outer and nested shapes
           |
           v
  Nested and starred unpacking
           |
           v
  Domain and numerical validation
           |
           v
 Immutable normalized trade fills
           |
           v
  Stage asset and venue summaries
           |
           v
 Verify accounting identities
           |
           v
 Publish immutable reconciliation
```

### Financial Semantics

- **Gross notional:** `quantity * execution_price` (USD, nonnegative).
- **Signed quantity:** `+quantity` for BUY and `-quantity` for SELL.
- **Cash flow for BUY:** `-(quantity * price + fee)`.
- **Cash flow for SELL:** `quantity * price - fee`.
- **Aggregated identity:** `net_cash_flow = sell_notional - buy_notional - total_fees`.
- **Gross identity:** `gross_notional = buy_notional + sell_notional`.

These quantities describe **execution tape flows**. Net quantity does not establish ownership of a security, cost basis, realized P&L, or NAV without opening balances and additional rules. It is valid for tape reconciliation to show a negative net quantity even though the case study itself is not a short-selling engine.

### Design Requirements

1. Standard library only, including `decimal.Decimal`.
2. Bounded event count; no infinite-stream materialization.
3. Reject duplicate event IDs and invalid nested shapes.
4. Preserve any optional metadata tags in an immutable tuple.
5. No mutation of caller-owned input records.
6. Validate all records before returning the authoritative report.
7. Produce deterministic ordering for asset and venue summaries.
8. Implement independent financial reconciliation tests.
9. Clearly distinguish data-shape checks from domain validation.
10. Document operational limitations (no durable log, broker reconciliation, or exchange connectivity).

In [12]:
from dataclasses import dataclass
from decimal import Decimal
from typing import Literal


TradeSide = Literal["BUY", "SELL"]
ZERO_USD = Decimal("0.00")
ALLOWED_VENUES = frozenset({"NYSE", "NASDAQ", "IEX"})


@dataclass(frozen=True, slots=True)
class NormalizedFill:
    """A complete, validated execution-tape event."""

    event_id: str
    ticker: str
    side: TradeSide
    quantity: int
    price_usd: Decimal
    fee_usd: Decimal
    venue: str
    tags: tuple[str, ...]

    @property
    def notional_usd(self) -> Decimal:
        """Return the absolute execution notional in USD."""
        return self.price_usd * self.quantity

    @property
    def signed_quantity(self) -> int:
        """Return positive quantity for BUY, negative for SELL."""
        return self.quantity if self.side == "BUY" else -self.quantity

    @property
    def cash_flow_usd(self) -> Decimal:
        """Return signed execution cash flow net of fees."""
        if self.side == "BUY":
            return -(self.notional_usd + self.fee_usd)
        return self.notional_usd - self.fee_usd


@dataclass(frozen=True, slots=True)
class AssetReconciliation:
    """Asset-level signed quantity and financial flow summary."""

    ticker: str
    net_quantity: int
    gross_notional_usd: Decimal
    total_fee_usd: Decimal
    net_cash_flow_usd: Decimal


@dataclass(frozen=True, slots=True)
class TapeReconciliation:
    """Immutable summary of a validated finite trade-event tape."""

    fills: tuple[NormalizedFill, ...]
    assets: tuple[AssetReconciliation, ...]
    venue_notionals: tuple[tuple[str, Decimal], ...]
    buy_notional_usd: Decimal
    sell_notional_usd: Decimal
    gross_notional_usd: Decimal
    total_fees_usd: Decimal
    net_cash_flow_usd: Decimal


print("Execution tape data models initialized.")

Execution tape data models initialized.


### **Analytical Interpretation**

**1. Domain Objects.** `NormalizedFill` stores a validated event in a named, immutable structure rather than exposing ambiguous positional fields throughout the computation.

**2. Directionality.** Signed quantities and cash flows are calculated using the trade side; buying consumes cash while selling generates cash before fees.

**3. Exact Monetary Representation.** `Decimal` preserves canonical cent-denominated input values without binary floating-point conversion. This is important for financial reconciliation, although production systems still need explicit precision and settlement rules.

**4. Immutable Reports.** Asset and venue summaries use tuples and frozen dataclasses, making accidental post-publication mutation through their ordinary public interfaces harder.

**5. Scope Limitation.** A tape-level net quantity or net cash flow is not a portfolio balance sheet or realized profit calculation.

In [13]:
import re
from collections.abc import Sequence
from decimal import InvalidOperation

MONEY_PATTERN = re.compile(r"(?:0|[1-9][0-9]{0,8})\.[0-9]{2}\Z")


def parse_usd(value: object, *, positive: bool) -> Decimal:
    """Parse a canonical cent-denominated USD string."""
    if not isinstance(value, str) or MONEY_PATTERN.fullmatch(value) is None:
        raise ValueError("USD amounts must have canonical xx.yy form")
    try:
        result = Decimal(value)
    except InvalidOperation as exc:
        raise ValueError("Invalid USD amount") from exc
    if not result.is_finite() or (result <= 0 if positive else result < 0):
        raise ValueError("USD amount outside permitted range")
    return result


def parse_execution_record(record: Sequence[object]) -> NormalizedFill:
    """Parse a four-field nested event plus zero to three optional tags.

    The nested starred-target statement ensures structural validation.
    The entire result is validated before becoming a NormalizedFill.
    """
    if isinstance(record, (str, bytes)) or not isinstance(record, Sequence):
        raise TypeError("Each event must be a finite sequence")
    if not 4 <= len(record) <= 7:
        raise ValueError("Expected four required fields and up to three tags")

    try:
        (
            event_id,
            (ticker, side),
            (quantity, price_text),
            (fee_text, venue),
            *tags,
        ) = record
    except (ValueError, TypeError) as exc:
        raise ValueError("Invalid nested execution-record shape") from exc

    if not isinstance(event_id, str) or not event_id.strip():
        raise ValueError("event_id must be a nonempty string")
    if not isinstance(ticker, str) or not ticker.isalpha() or not ticker.isupper():
        raise ValueError("ticker must be an uppercase alphabetic symbol")
    if side not in ("BUY", "SELL") or not isinstance(side, str):
        raise ValueError("side must be BUY or SELL")
    if type(quantity) is not int or not 1 <= quantity <= 1_000_000:
        raise ValueError("quantity must be an integer between 1 and 1,000,000")
    if not isinstance(venue, str) or venue not in ALLOWED_VENUES:
        raise ValueError("unsupported trading venue")
    if not all(isinstance(tag, str) and tag.strip() for tag in tags):
        raise ValueError("tags must be nonempty strings")
    if len(set(tags)) != len(tags):
        raise ValueError("duplicate optional tag")

    return NormalizedFill(
        event_id=event_id,
        ticker=ticker,
        side=side,
        quantity=quantity,
        price_usd=parse_usd(price_text, positive=True),
        fee_usd=parse_usd(fee_text, positive=False),
        venue=venue,
        tags=tuple(tags),
    )


sample = (
    "E001", ("AAPL", "BUY"), (100, "180.00"),
    ("1.25", "NYSE"), "opening", "momentum"
)
parsed = parse_execution_record(sample)
assert parsed.tags == ("opening", "momentum")
assert parsed.notional_usd == Decimal("18000.00")
assert parsed.cash_flow_usd == Decimal("-18001.25")
print("Parsed event:", parsed)

Parsed event: NormalizedFill(event_id='E001', ticker='AAPL', side='BUY', quantity=100, price_usd=Decimal('180.00'), fee_usd=Decimal('1.25'), venue='NYSE', tags=('opening', 'momentum'))


### **Analytical Interpretation**

**1. Nested Destructuring.** The parser uses one structural unpacking statement to extract the mandatory fields and capture the optional suffix as a list.

**2. Explicit Schema Constraints.** Bounds on outer length and nested arity prevent malformed records from being silently accepted.

**3. Separation of Syntax and Domain.** Successful unpacking is followed by validations of event IDs, side, quantity, venue, tags, and canonical monetary strings.

**4. Decimal Precision.** Strict cent-formatted strings become exact `Decimal` values, eliminating an avoidable binary-float conversion during parsing.

**5. Publication Boundary.** Only a completely validated immutable `NormalizedFill` is returned; no authoritative portfolio or tape state is modified by the parser.

**6. Important Limitation.** Input sequences are assumed not to mutate concurrently during parsing; a fully adversarial or concurrent ingestion environment needs stronger boundary controls.

In [14]:
from collections.abc import Iterable


def reconcile_execution_tape(
    raw_records: Iterable[Sequence[object]],
    *,
    max_events: int = 10_000,
) -> TapeReconciliation:
    """Normalize and reconcile a finite execution-tape stream.

    The iterator is consumed once. Rejected inputs produce no report,
    but consumption of the source iterable is not rolled back.
    """
    if type(max_events) is not int or not 1 <= max_events <= 100_000:
        raise ValueError("max_events outside permitted range")

    normalized: list[NormalizedFill] = []
    seen_ids: set[str] = set()

    # Stage and validate the entire bounded event collection.
    for index, raw_record in enumerate(raw_records, start=1):
        if index > max_events:
            raise ValueError("Execution tape exceeds max_events")
        fill = parse_execution_record(raw_record)
        if fill.event_id in seen_ids:
            raise ValueError(f"Duplicate event ID: {fill.event_id}")
        seen_ids.add(fill.event_id)
        normalized.append(fill)

    if not normalized:
        raise ValueError("Execution tape cannot be empty")

    # Keys are tickers and values are mutable staging accumulators.
    # Published results below are new immutable records.
    assets: dict[str, dict[str, object]] = {}
    venue_notionals: dict[str, Decimal] = {}
    buy_notional = ZERO_USD
    sell_notional = ZERO_USD
    total_fees = ZERO_USD
    net_cash_flow = ZERO_USD

    for fill in normalized:
        notional, cash_flow = fill.notional_usd, fill.cash_flow_usd
        if fill.side == "BUY":
            buy_notional += notional
        else:
            sell_notional += notional
        total_fees += fill.fee_usd
        net_cash_flow += cash_flow

        if fill.ticker not in assets:
            assets[fill.ticker] = {
                "net_quantity": 0,
                "notional": ZERO_USD,
                "fees": ZERO_USD,
                "cash_flow": ZERO_USD,
            }
        state = assets[fill.ticker]
        state["net_quantity"] += fill.signed_quantity
        state["notional"] += notional
        state["fees"] += fill.fee_usd
        state["cash_flow"] += cash_flow
        venue_notionals[fill.venue] = (
            venue_notionals.get(fill.venue, ZERO_USD) + notional
        )

    gross_notional = buy_notional + sell_notional
    assert net_cash_flow == sell_notional - buy_notional - total_fees
    assert sum(venue_notionals.values(), ZERO_USD) == gross_notional

    asset_rows = tuple(
        AssetReconciliation(
            ticker=ticker,
            net_quantity=state["net_quantity"],
            gross_notional_usd=state["notional"],
            total_fee_usd=state["fees"],
            net_cash_flow_usd=state["cash_flow"],
        )
        for ticker, state in sorted(assets.items())
    )
    assert sum((r.net_cash_flow_usd for r in asset_rows), ZERO_USD) == net_cash_flow

    return TapeReconciliation(
        fills=tuple(normalized),
        assets=asset_rows,
        venue_notionals=tuple(sorted(venue_notionals.items())),
        buy_notional_usd=buy_notional,
        sell_notional_usd=sell_notional,
        gross_notional_usd=gross_notional,
        total_fees_usd=total_fees,
        net_cash_flow_usd=net_cash_flow,
    )


print("Execution-tape reconciliation engine initialized.")

Execution-tape reconciliation engine initialized.


### **Analytical Interpretation**

**1. Streaming Ingestion.** The function accepts an iterable and processes it once, with an explicit maximum number of records to limit resource usage.

**2. Staged Validation.** Event shapes, numerical values, and duplicate IDs are validated before the report is published; a failure raises rather than yielding an incomplete report.

**3. Local Mutability.** Intermediate asset summaries use mutable dictionaries for efficient aggregation, but these are not exposed in the immutable report.

**4. Multiple Assignment and Named Values.** Temporary calculations such as `notional, cash_flow = ...` bind related values without changing an external ledger.

**5. Reconciliation Invariants.** The reported buy/sell notional, fees, cash flow, asset totals, and venue totals satisfy independent accounting identities.

**6. Failure Semantics.** Failure prevents report publication, **not** source-iterator consumption. This is not a durable, all-or-nothing message-queue transaction.

**7. Financial Boundary.** Execution flows are not equivalent to realized P&L or portfolio NAV.

In [15]:
# ============================================================
# SYNTHETIC NESTED EXECUTION TAPE
# ============================================================

raw_tape = [
    ("E001", ("AAPL", "BUY"), (100, "180.00"),
     ("1.25", "NYSE"), "opening", "momentum"),
    ("E002", ("MSFT", "BUY"), (50, "400.00"),
     ("2.50", "NASDAQ"), "hedge"),
    ("E003", ("AAPL", "SELL"), (40, "190.00"),
     ("1.00", "NYSE"), "rebalance", "approved"),
    ("E004", ("JPM", "BUY"), (30, "150.00"),
     ("1.25", "NYSE")),
]

# Pass a generator to demonstrate single-pass ingestion.
report = reconcile_execution_tape((event for event in raw_tape))

print("=" * 78)
print("NESTED EXECUTION TAPE RECONCILIATION")
print("=" * 78)
print(f"{'ID':<7}{'Ticker':<9}{'Side':<7}{'Qty':>7}{'Notional USD':>17}{'Flow USD':>17}")
print("-" * 78)
for fill in report.fills:
    print(
        f"{fill.event_id:<7}{fill.ticker:<9}{fill.side:<7}"
        f"{fill.quantity:>7}{fill.notional_usd:>17,.2f}"
        f"{fill.cash_flow_usd:>17,.2f}"
    )

print("\nASSET RECONCILIATION")
print("-" * 78)
for asset in report.assets:
    print(
        f"{asset.ticker:<8} Net shares: {asset.net_quantity:>5}  "
        f"Cash flow: ${asset.net_cash_flow_usd:>12,.2f}"
    )

print("\nVENUE NOTIONALS")
for venue, notional in report.venue_notionals:
    print(f"{venue:<9}${notional:,.2f}")

print("\nACCOUNTING TOTALS")
print(f"Buy notional:   ${report.buy_notional_usd:,.2f}")
print(f"Sell notional:  ${report.sell_notional_usd:,.2f}")
print(f"Gross notional: ${report.gross_notional_usd:,.2f}")
print(f"Total fees:     ${report.total_fees_usd:,.2f}")
print(f"Net cash flow:  ${report.net_cash_flow_usd:,.2f}")

assert len(report.fills) == 4
assert report.buy_notional_usd == Decimal("42500.00")
assert report.sell_notional_usd == Decimal("7600.00")
assert report.gross_notional_usd == Decimal("50100.00")
assert report.total_fees_usd == Decimal("6.00")
assert report.net_cash_flow_usd == Decimal("-34906.00")
assert {x.ticker: x.net_quantity for x in report.assets} == {
    "AAPL": 60, "JPM": 30, "MSFT": 50
}

NESTED EXECUTION TAPE RECONCILIATION
ID     Ticker   Side       Qty     Notional USD         Flow USD
------------------------------------------------------------------------------
E001   AAPL     BUY        100        18,000.00       -18,001.25
E002   MSFT     BUY         50        20,000.00       -20,002.50
E003   AAPL     SELL        40         7,600.00         7,599.00
E004   JPM      BUY         30         4,500.00        -4,501.25

ASSET RECONCILIATION
------------------------------------------------------------------------------
AAPL     Net shares:    60  Cash flow: $  -10,402.25
JPM      Net shares:    30  Cash flow: $   -4,501.25
MSFT     Net shares:    50  Cash flow: $  -20,002.50

VENUE NOTIONALS
NASDAQ   $20,000.00
NYSE     $30,100.00

ACCOUNTING TOTALS
Buy notional:   $42,500.00
Sell notional:  $7,600.00
Gross notional: $50,100.00
Total fees:     $6.00
Net cash flow:  $-34,906.00



### Expected Results

#### Event-Level Reconciliation

| Event | Ticker | Side | Quantity | Gross Notional | Net Cash Flow |
|---|---|---|---:|---:|---:|
| E001 | AAPL | BUY | 100 | USD 18,000.00 | -USD 18,001.25 |
| E002 | MSFT | BUY | 50 | USD 20,000.00 | -USD 20,002.50 |
| E003 | AAPL | SELL | 40 | USD 7,600.00 | +USD 7,599.00 |
| E004 | JPM | BUY | 30 | USD 4,500.00 | -USD 4,501.25 |

#### Aggregated Metrics

| Metric | Expected Value |
|---|---:|
| Validated Execution Events | 4 |
| Buy Notional | USD 42,500.00 |
| Sell Notional | USD 7,600.00 |
| Gross Notional | USD 50,100.00 |
| Total Fees | USD 6.00 |
| Net Cash Flow | -USD 34,906.00 |
| AAPL Net Executed Shares | +60 |
| JPM Net Executed Shares | +30 |
| MSFT Net Executed Shares | +50 |
| NASDAQ Notional | USD 20,000.00 |
| NYSE Notional | USD 30,100.00 |


**Identity:** `7,600.00 − 42,500.00 − 6.00 = −34,906.00` (USD).

**Interpretation:** This is the cash movement implied by these four executions, **not** a profit, loss, equity, portfolio cash balance, or buying-power calculation.

### **Analytical Interpretation**

**1. Normalized Records.** The four heterogeneous nested tuples, including different lengths of optional tag metadata, are normalized into a consistent immutable event representation.

**2. Extended Unpacking.** Starred suffix capture retains zero, one, or multiple optional tags without weakening the mandatory nested-field schema.

**3. Exact Arithmetic.** Monetary values are parsed from canonical strings into `Decimal` and reconciled without binary floating-point rounding.

**4. Signed Financial Semantics.** Buy events have negative cash flows; sells have positive cash flows before commission deductions.

**5. Multi-Level Reconciliation.** Asset-level cash flows aggregate to the report total, and venue notionals sum to gross execution notional.

**6. Data Integrity.** The parser does not modify the original `raw_tape`, and the returned report uses immutable records.

**7. Replay Limitation.** Replaying a generator requires a new iterator or durable event store. This notebook does not persist messages or provide exactly-once delivery semantics.

**8. Scope Discipline.** Net executed quantities and cash flows are **not** realized trading P&L; no opening holdings or cost basis have been supplied.

## 2.4.13 Automated Testing and Financial Invariants

### Testing Objectives

The test suite verifies both Python unpacking mechanics and the financial contracts of the trade-tape processor.

**Structural tests:** exact arity, optional tags, invalid nested shapes, incorrect source type, duplicate tags, strict canonical money representation, generator consumption, and single-pass processing.

**Financial tests:** valid BUY/SELL directionality, duplicate event IDs, total notional identity, net cash-flow identity, per-asset/venue aggregation, invalid price and fee, input preservation, and deterministic reconstruction.

**Rollback distinction:** a failed batch publishes no partial report; the source iterator may still be consumed. This is not equivalent to a durable broker or database transaction.

**Numerical invariant:**

\[
\text{NetCashFlow}=\text{SellNotional}-\text{BuyNotional}-\text{Fees}
\]

All production-facing inputs should additionally be checked against provider schemas, execution timestamps, trade corrections, time zones, market conventions, and operational resource limits.

In [16]:
import unittest
from copy import deepcopy
from dataclasses import FrozenInstanceError
from decimal import Decimal
from random import Random


class UnpackingAndTapeTests(unittest.TestCase):
    """Test advanced unpacking and execution-tape reconciliation."""

    def setUp(self):
        self.raw = deepcopy(raw_tape)
        self.report = reconcile_execution_tape(self.raw)

    def test_reference_totals(self):
        self.assertEqual(self.report.gross_notional_usd, Decimal("50100.00"))
        self.assertEqual(self.report.net_cash_flow_usd, Decimal("-34906.00"))

    def test_canonical_decimal_accounting(self):
        self.assertEqual(
            self.report.sell_notional_usd
            - self.report.buy_notional_usd
            - self.report.total_fees_usd,
            self.report.net_cash_flow_usd,
        )

    def test_asset_aggregation(self):
        self.assertEqual(
            sum((a.net_cash_flow_usd for a in self.report.assets), ZERO_USD),
            self.report.net_cash_flow_usd,
        )
        self.assertEqual(
            {a.ticker: a.net_quantity for a in self.report.assets},
            {"AAPL": 60, "JPM": 30, "MSFT": 50},
        )

    def test_venue_aggregation(self):
        self.assertEqual(
            sum((v for _, v in self.report.venue_notionals), ZERO_USD),
            self.report.gross_notional_usd,
        )

    def test_starred_optional_tags(self):
        self.assertEqual(self.report.fills[0].tags, ("opening", "momentum"))
        self.assertEqual(self.report.fills[-1].tags, ())

    def test_invalid_nested_shape(self):
        bad = ("BAD", ("AAPL", "BUY"), (100,), ("1.00", "NYSE"))
        with self.assertRaises(ValueError):
            parse_execution_record(bad)

    def test_too_few_fields(self):
        with self.assertRaises(ValueError):
            parse_execution_record(("E", ("AAPL", "BUY")))

    def test_too_many_tags(self):
        bad = self.raw[0] + ("extra1", "extra2")
        with self.assertRaises(ValueError):
            parse_execution_record(bad)

    def test_duplicate_tags(self):
        bad = ("E5", ("AAPL", "BUY"), (1, "10.00"),
               ("0.00", "NYSE"), "dup", "dup")
        with self.assertRaises(ValueError):
            parse_execution_record(bad)

    def test_boolean_quantity(self):
        bad = ("E5", ("AAPL", "BUY"), (True, "10.00"),
               ("0.00", "NYSE"))
        with self.assertRaises(ValueError):
            parse_execution_record(bad)

    def test_invalid_money_strings(self):
        for price in ("NaN", "1.2", "1e2", "-1.00", "00.01"):
            with self.subTest(price=price):
                bad = ("E5", ("AAPL", "BUY"), (1, price),
                       ("0.00", "NYSE"))
                with self.assertRaises(ValueError):
                    parse_execution_record(bad)

    def test_negative_fee(self):
        bad = ("E5", ("AAPL", "BUY"), (1, "10.00"),
               ("-0.01", "NYSE"))
        with self.assertRaises(ValueError):
            parse_execution_record(bad)

    def test_duplicate_event_id(self):
        with self.assertRaisesRegex(ValueError, "Duplicate event ID"):
            reconcile_execution_tape([self.raw[0], self.raw[0]])

    def test_stream_is_consumed_once(self):
        stream = (event for event in self.raw)
        result = reconcile_execution_tape(stream)
        self.assertEqual(len(result.fills), 4)
        self.assertEqual(list(stream), [])

    def test_event_limit(self):
        with self.assertRaises(ValueError):
            reconcile_execution_tape(self.raw, max_events=3)

    def test_immutable_published_record(self):
        with self.assertRaises(FrozenInstanceError):
            self.report.fills[0].ticker = "JPM"

    def test_input_preservation(self):
        original = deepcopy(self.raw)
        reconcile_execution_tape(self.raw)
        self.assertEqual(self.raw, original)

    def test_deterministic_reconstruction(self):
        again = reconcile_execution_tape(deepcopy(raw_tape))
        self.assertEqual(self.report, again)

    def test_empty_tape(self):
        with self.assertRaises(ValueError):
            reconcile_execution_tape([])

    def test_seeded_reconciliation_invariants(self):
        rng = Random(2026)
        for index in range(60):
            quantity = rng.randint(1, 500)
            price_cents = rng.randint(100, 80_000)
            fee_cents = rng.randint(0, 1000)
            price = f"{price_cents // 100}.{price_cents % 100:02d}"
            fee = f"{fee_cents // 100}.{fee_cents % 100:02d}"
            side = "BUY" if index % 2 == 0 else "SELL"
            raw = (f"S{index:03d}", ("AAPL", side),
                   (quantity, price), (fee, "IEX"))
            rec = reconcile_execution_tape([raw])
            self.assertEqual(
                rec.net_cash_flow_usd,
                rec.sell_notional_usd
                - rec.buy_notional_usd
                - rec.total_fees_usd,
            )


suite = unittest.defaultTestLoader.loadTestsFromTestCase(
    UnpackingAndTapeTests
)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()

test_asset_aggregation (__main__.UnpackingAndTapeTests.test_asset_aggregation) ... 

ok


test_boolean_quantity (__main__.UnpackingAndTapeTests.test_boolean_quantity) ... 

ok


test_canonical_decimal_accounting (__main__.UnpackingAndTapeTests.test_canonical_decimal_accounting) ... 

ok


... (31 outputs left out)

ok


test_venue_aggregation (__main__.UnpackingAndTapeTests.test_venue_aggregation) ... 

ok


----------------------------------------------------------------------
Ran 20 tests in 0.014s

OK


### **Analytical Interpretation**

**1. Structural Validation.** The tests reject missing required fields, improperly nested data, excessive tags, and duplicate tag names.

**2. Exact Monetary Data.** Invalid price encodings and negative fees are rejected before they can enter accounting calculations.

**3. Directional Accounting.** Buy and sell notionals, fees, and net cash flows satisfy the stated signed-flow identity.

**4. Multi-Level Aggregation.** Independently aggregated assets and venues reconcile to the global report totals.

**5. Iterator Exhaustion.** The one-pass ingestion test confirms that the generator cannot be replayed without reconstruction.

**6. Publication Safety.** Failing records produce exceptions rather than partially published reports; source consumption is still irreversible.

**7. Determinism and Property-Oriented Coverage.** Seeded randomized examples verify the main accounting identity over many valid inputs; this does not constitute exhaustive property-based testing.

**8. Financial Boundary.** These tests validate tape flows and schema contracts, not clearing, settlement, market impact, NAV, or trading profitability.

## 2.4.14 Final Analytical Interpretation and Engineering Trade-offs

### 1. Iterables, Not Just Sequences

Python assignment destructuring works with general iterables. The runtime need not obtain values through integer indexing, so iterator consumption is a first-class part of the semantics.

### 2. Exact vs. Extended Arity

Fixed-size unpacking enforces exact length; starred targets permit variable-length capture subject to required prefix and suffix fields. A starred target receives a list, including when the remainder is empty.

### 3. Recursive Assignment

Nested targets unpack recursively. Earlier targets may already be assigned when a later nested unpack fails; there is no universal automatic rollback.

### 4. User-Defined Side Effects

Custom iterators, attributes, and subscription targets can execute arbitrary supported Python behavior during unpacking. Do not assume destructuring is a purely declarative transformation.

### 5. Data-Source Protocols

Generators are single-pass sources. Unpacking may consume extra data to detect a mismatch; starred capture can materialize a large remainder. Continuous feeds should be converted into bounded events at an appropriate boundary.

### 6. Mapping and String Semantics

A mapping yields keys when iterated; a string yields code points. Neither operation automatically matches the financial business meaning of a record.

### 7. Starred Target vs. Expansion

A starred target captures a list; starred expressions expand iterables into calls or displays. Their similar syntax does not imply identical operational or copying semantics.

### 8. Structural Pattern Matching

`match`/`case` follows distinct subject eligibility and failure rules. It is not a drop-in replacement for assignment unpacking or financial schema validation.

### 9. Static Inspection

AST and bytecode reveal structural differences between fixed and extended unpacking. They do not prove runtime shapes, type correctness, or the absence of side effects.

### 10. Type and Schema Contracts

Typed tuples and named records improve documentation, but runtime input validation remains necessary when ingesting external trading data. Shape correctness is not financial correctness.

### 11. Exact Financial Arithmetic

Cent-denominated `Decimal` values avoid avoidable binary-float conversion in the practical application. Real production systems must still define supported currencies, rounding, corrections, precision, and settlement rules.

### 12. Reconciliation vs. Profitability

The tape processor correctly computes executed quantity, gross notional, fees, and signed cash flow. Without opening positions and accounting rules, these values cannot establish realized P&L or NAV.

### 13. Failure and Transaction Scope

A failed event sequence produces no report, but previously consumed generator values are not restored. Durable all-or-nothing broker ingestion requires an external persistence and acknowledgment protocol.

### 14. Production Extensions

Further work would add event timestamps, trade cancels/corrections, broker/exchange reconciliation, idempotency keys, durable storage, structured audit logs, bounded queue semantics, currency normalization, and event schema versioning.

### Final Assessment

Mastery of sequence unpacking means understanding **iteration, arity, nested target order, starred capture, failure boundaries, and domain validation**. The core quantitative engineering skill is transforming variable-shaped external data into explicit, validated, reproducible records before financial calculations rely on it.

## Advanced Practice Exercises — Expert Level

**Instructions:** Work independently. For each exercise, present (a) a documented interface, (b) deterministic code, (c) at least one failing case, (d) assertions or unit tests, and (e) a short **Analytical Interpretation** discussing language semantics and financial implications.

### Exercise 1 — Exact Arity and Iterator Consumption Audit

Implement a custom iterable that logs `__iter__()` and `__next__()` calls. Demonstrate successful exact unpacking, insufficient values, and excess values. Explain which values have been consumed after each failure. Avoid assuming an exact interpreter-specific instruction sequence.

### Exercise 2 — Nested Financial Record Validation

Design a schema with nested records for `(execution_id, (ticker, side), (quantity, price), (fee, venue))`. Accept tuples, lists, and finite iterables where your contract permits them. Distinguish structural failures from invalid prices and duplicate IDs. Ensure validation occurs before publishing a normalized record.

### Exercise 3 — Variable-Length Trading Metadata

Add optional trailing tags, one optional audit code, and a version identifier. Implement a backward-compatible schema migration from a four-field event format to a richer version. Document how you distinguish schema versions without guessing from ambiguous values.

### Exercise 4 — Nested Assignment Failure Matrix

Construct at least six cases covering top-level mismatches, nested mismatches, custom iterator exceptions, attribute setter failures, subscription setter failures, and iterators with side effects. Record the state that survives failure and explain why ordinary Python assignment does not provide rollback guarantees.

### Exercise 5 — Structural Pattern Matching Comparison

Implement the same record classifier once using unpacking and once using `match`/`case`. Compare eligible input types, shape mismatch behavior, guards, variable bindings, and fallback logic. Include a generator, a string, a tuple, and a list as subjects.

### Exercise 6 — High-Volume Streaming Adapter

Build a bounded ingestion adapter using `itertools.islice`, `enumerate`, and explicit event caps. Reject overlong records and oversized batches. Measure peak memory of different parsing choices with `tracemalloc`; do not claim `*rest` is safe for unbounded streams.

### Exercise 7 — AST-Based Unpacking Linter

Parse `.py` source without executing it and find `ast.Assign` nodes containing tuple/list destructuring, `ast.Starred` targets, and nested targets with attribute or subscript writes. Report source line numbers and selected potential partial-update risks. Discuss why AST alone cannot determine whether iterators are finite.

### Exercise 8 — Financial Reconciliation with `Decimal`

Extend the execution-tape project to handle multiple market venues and partial execution batches. Validate price and fee formatting, report buy/sell notional and fees by asset, and independently verify every signed cash-flow identity. Keep realized P&L out of scope unless opening holdings and cost-basis conventions are defined.

### Exercise 9 — Failure-Safe Pipeline Publication

Construct a pipeline that parses a finite batch into immutable events, validates every event, calculates preliminary reconciliations, and publishes a new result object only on success. Demonstrate that a failed source iterator can still be consumed, and design a separate durable replay strategy.

### Exercise 10 — Type-Driven Schema Design

Compare heterogeneous tuples, `NamedTuple`, frozen dataclasses, and explicitly versioned record objects. Evaluate type-checker usefulness, schema evolution, readability, memory behavior, and runtime validation costs. Recommend one representation for a real-time order event and one for a persisted audit record.

## Advanced Challenge — Schema-Aware, Replayable Market Event Infrastructure

### Objective

Develop a modular ingestion and reconciliation platform for variable-length, nested market/execution events. The platform must demonstrate rigorous iterable-unpacking semantics while preserving operational auditability and monetary correctness.

### Components

**1. Protocol and Schema Layer** — Versioned event grammars, nested shape constraints, optional tagged metadata, explicit money/currency conventions, and typed normalized records.

**2. One-Pass Ingestion** — Finite event adapters, controlled iterator consumption, hard resource limits, explicit detection of overlong records, and error classification by processing stage.

**3. Validation and Normalization** — Strict ticker, side, quantity, price, fee, and venue contracts; canonical `Decimal` parsing; duplicate ID handling; documented data corrections.

**4. Reconciliation** — Asset-level signed quantities, venue-level notionals, fee aggregation, signed cash-flow identities, and independent reconciliation checks.

**5. Replay and Durability** — Persistent event log, deterministic replay, idempotent event IDs, correction/cancel events, versioned outputs, and clearly defined recovery/acknowledgment semantics. These are extensions, not guarantees of the notebook implementation.

**6. Code Quality and Audit** — AST-based unpacking analysis, static typing, linting, unit testing, seed-controlled randomized tests, property-oriented invariants, and CI checks.

### Technical Requirements

1. Python 3.10+; identify implementation-specific behavior explicitly.
2. Use only validated events to build reconciliations.
3. Do not use unrestricted `eval()` or `exec()` to parse records.
4. Support nested exact and starred assignment targets and document their failure boundaries.
5. Never assume a source iterator can be replayed.
6. Avoid using a starred target to consume an unbounded feed.
7. Use exact decimal arithmetic for the defined monetary operations.
8. Distinguish tape cash flows from profit, NAV, or available cash.
9. Maintain explicit event and schema versioning.
10. Verify end-to-end accounting and data-integrity invariants.

### Expected Deliverables

- Parsing and normalization modules.
- Immutable event and reconciliation data models.
- Bounded stream adapter and documented failure semantics.
- Decimal financial calculation and aggregation engine.
- Deterministic replay and idempotency design.
- AST-based unpacking auditor.
- Unit, randomized, and regression test suite.
- Reproducible environment configuration and CI workflow.
- A technical discussion of performance, resource limits, and security assumptions.

### Evaluation Criteria

| Dimension | Weight |
|---|---:|
| Unpacking semantics and protocol correctness | 25% |
| Financial reconciliation correctness | 25% |
| Data contracts, validation, and boundaries | 20% |
| Testing and reproducibility | 15% |
| Architecture and maintainability | 10% |
| Documentation and auditability | 5% |

### Final Goal

Transform flexible Python destructuring syntax into an explicit, auditable market-data engineering workflow without confusing source-code convenience with business-schema correctness or transactional guarantees.

## Notebook Summary and Research Checklist

| Section | Topic | Main outcome |
|---|---|---|
| 2.4.1 | Formal semantics | Unpacking is iteration-based assignment |
| 2.4.2 | Iterator protocol | Arity checks consume iterators |
| 2.4.3 | Nested targets | Recursive unpacking with target side effects |
| 2.4.4 | Extended unpacking | Starred target captures a list |
| 2.4.5 | Lazy sources | One-shot consumption and bounded adapters |
| 2.4.6 | Mappings and loops | Keys, `enumerate`, `zip(strict=True)` |
| 2.4.7 | Expansion contexts | Assignment capture vs. call/display expansion |
| 2.4.8 | Pattern matching | Different subject and failure semantics |
| 2.4.9 | Failure boundaries | Nested partial writes; no automatic rollback |
| 2.4.10 | AST and bytecode | Structural inspection and portability limits |
| 2.4.11 | Contracts and pitfalls | Explicit domain validation and shape guarantees |
| 2.4.12 | Quant application | Nested execution-tape reconciliation |
| 2.4.13 | Automated testing | Schema, one-pass, and accounting invariants |
| 2.4.14 | Final interpretation | Engineering trade-offs and production limits |

### Completion Criteria

- [ ] Can explain what a general iterable contributes to unpacking.
- [ ] Can predict exact-length and starred-target behavior.
- [ ] Can identify partial assignment and iterator-consumption failures.
- [ ] Can distinguish unpacking from `match` and call expansion.
- [ ] Can inspect AST targets and explain bytecode portability limits.
- [ ] Can reproduce and independently validate the execution-tape totals.
- [ ] Can run the entire notebook from a clean kernel with all tests passing.
- [ ] Can explain why net cash flow is **not** realized trading profit.